# Gold Mauá Planejamento Urbano

Lê os 25 parquets silver do Lakehouse, aplica os mesmos tratamentos do fluxo local e grava a tabela **gold_maua_pl_urbano** no Lakehouse.

In [1]:
import pandas as pd
import re

# Caminhos no Fabric (Lakehouse default) – leitura silver
BASE_SAVE_PATH = "/lakehouse/default/Files/"
SUBPasta_SILVER = "silver_planejamento_urbano"
BASE_SILVER = f"{BASE_SAVE_PATH}{SUBPasta_SILVER}/"

# Nome da tabela gold (será gravada em Tables do lakehouse)
NOME_TABELA_GOLD = "gold_maua_pl_urbano"

print(f"Silver: {BASE_SILVER}")
print(f"Tabela gold: {NOME_TABELA_GOLD}")

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 3, Finished, Available, Finished, False)

Silver: /lakehouse/default/Files/silver_planejamento_urbano/
Tabela gold: gold_maua_pl_urbano


In [ ]:
%run ./nb_utils_maua_ingest_acto_gestao

## 1. Carregar as 25 tabelas silver do Lakehouse

In [2]:
# Ler silver_solicitacoes_maua1.parquet ... maua25.parquet do Lakehouse
list_dfs = []
for i in range(1, 26):
    path = f"{BASE_SILVER}silver_solicitacoes_maua{i}.parquet"
    try:
        d = pd.read_parquet(path)
        d["origem_num"] = i
        list_dfs.append(d)
    except Exception as e:
        print(f"Aviso: {path} -> {e}")

df = pd.concat(list_dfs, ignore_index=True, join="outer")
print(f"Arquivos lidos: {len(list_dfs)}")
print(f"Shape consolidado: {df.shape[0]} linhas | {df.shape[1]} colunas")

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 4, Finished, Available, Finished, False)

Arquivos lidos: 25
Shape consolidado: 7451 linhas | 249 colunas


## 2. Funções de tratamento

Remoção de pipe e número, dois pontos; bfill para unificar colunas duplicadas; snake_case; conversão de datas; opcional dropar esparsas.

In [3]:
# def tratar_nome_colunas(df: pd.DataFrame) -> pd.DataFrame:
#     """Remove pipe e número do nome; remove ':' no final (ex: 'Bairro:' -> 'Bairro')."""
#     df = df.copy()
#     renomear = {}
#     for c in df.columns:
#         nome = str(c).strip()
#         m = re.match(r"^(.+)\|(\d+)$", nome)
#         if m:
#             nome = m.group(1).strip()
#         elif re.match(r"^(.+)\((\d+)\)$", nome):
#             m2 = re.match(r"^(.+)\((\d+)\)$", nome)
#             if m2:
#                 nome = m2.group(1).strip()
#         nome = nome.rstrip(":").strip()
#         renomear[c] = nome
#     df = df.rename(columns=renomear)
#     return df


# def colunas_para_snake_case(df: pd.DataFrame) -> pd.DataFrame:
#     """Converte nomes das colunas para snake_case."""
#     import unicodedata
#     def to_snake(s):
#         s = str(s).strip()
#         s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
#         s = re.sub(r"[^a-zA-Z0-9\s]", " ", s)
#         s = re.sub(r"\s+", "_", s).strip("_").lower()
#         return s or "unnamed"
#     return df.rename(columns={c: to_snake(c) for c in df.columns})


# def consolidar_conceito_bfill(df: pd.DataFrame) -> pd.DataFrame:
#     """Colunas com mesmo nome são consolidadas em uma só com bfill(axis=1)."""
#     from collections import defaultdict
#     grupos = defaultdict(list)
#     for i in range(len(df.columns)):
#         grupos[df.columns[i]].append(i)
#     first_idx = {}
#     for i in range(len(df.columns)):
#         nome = df.columns[i]
#         if nome not in first_idx:
#             first_idx[nome] = i
#     result = {}
#     for nome in sorted(first_idx.keys(), key=lambda n: first_idx[n]):
#         indices = grupos[nome]
#         if len(indices) == 1:
#             result[nome] = df.iloc[:, indices[0]]
#         else:
#             sub = df.iloc[:, indices].bfill(axis=1)
#             result[nome] = sub.iloc[:, 0]
#     return pd.DataFrame(result, index=df.index)


# def converter_colunas_data(df: pd.DataFrame, colunas: list | None = None) -> pd.DataFrame:
#     """Converte colunas de data para datetime (erros viram NaT)."""
#     df = df.copy()
#     colunas = colunas or [c for c in df.columns if c.startswith("data_")]
#     for c in colunas:
#         if c not in df.columns:
#             continue
#         df[c] = pd.to_datetime(df[c], errors="coerce")
#     return df


# def dropar_colunas_esparsas(df: pd.DataFrame, limite_nan_pct: float = 0.95) -> pd.DataFrame:
#     """Remove colunas com >= limite_nan_pct de nulos."""
#     df = df.copy()
#     n = len(df)
#     dropar = [c for c in df.columns if df[c].isna().sum() / n >= limite_nan_pct]
#     return df.drop(columns=dropar)

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 5, Finished, Available, Finished, False)

## 3. Pipeline de tratamento e escrita Gold

Aplica: tratar nomes (pipe, número, dois pontos) → snake_case → bfill (uma vez) → datas → [opcional] dropar esparsas. Grava **gold_maua_pl_urbano** no Lakehouse.

In [4]:
# Fluxo: tratar nomes → snake_case → bfill (uma vez) → datas → df_gold
df = tratar_nome_colunas(df)
df = colunas_para_snake_case(df)
df = consolidar_conceito_bfill(df)

COLUNAS_DATA = [
    "data_finalizacao", "data_criacao", "data_criacao_etapa",
    "data_de_emissao", "data_de_validade",
]
df = converter_colunas_data(df, [c for c in COLUNAS_DATA if c in df.columns])

# Opcional: dropar colunas com >= 95% nulos
# df = dropar_colunas_esparsas(df, limite_nan_pct=0.95)

df_gold = df
print(f"Gold: {len(df_gold)} linhas | {len(df_gold.columns)} colunas")

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 6, Finished, Available, Finished, False)

Gold: 7451 linhas | 50 colunas


In [5]:
df0 = pd.read_parquet(BASE_SILVER + "silver_solicitacoes_maua0.parquet")
print("Colunas:", list(df0.columns))
print("Total:", len(df0.columns))

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 7, Finished, Available, Finished, False)

Colunas: ['Nº Solicitação|1', 'Essa solicitação de Alvará de aprovação de projeto e execução para demolição será deferida, indeferida ou retornará para Análise Técnica?|1245', 'Essa solicitação deverá ser:|1973', 'Essa solicitação de certificado de conclusão será deferida, indeferida ou retornará para Análise Técnica?|1358', 'Decisão da gestão:|583', 'Essa solicitação de certificado de uso será deferida, indeferida ou retornará para Análise Técnica?|1135', 'Essa solicitação de Alvará de aprovação de projeto e execução para demolição será deferida, indeferida ou retornará para Análise Técnica?|1424', 'Essa solicitação de Alvará de aprovação de projeto e execução para movimentação de terra, será deferida, indeferida ou retornará para Análise Técnica?|1326', 'Essa solicitação de Alvará de aprovação de projeto de muro de arrimo será deferida, indeferida ou retornará para Análise Técnica?|1005', 'Essa solicitação de Alvará de aprovação de projeto e execução para reconstrução será deferida, 

# TABELA DESISÕES

In [6]:
# Carregar silver_solicitacoes_maua0 (tabela de decisões) e consolidar coluna Decisão
df_decisoes = pd.read_parquet(BASE_SILVER + "silver_solicitacoes_maua0.parquet")
colunas_decisao = [c for c in df_decisoes.columns if c != "Nº Solicitação|1"]
df_decisoes["Decisão"] = df_decisoes[colunas_decisao].bfill(axis=1).iloc[:, 0]
df_decisoes = df_decisoes[["Nº Solicitação|1", "Decisão"]].copy()
df_decisoes.columns = ["no_solicitacao", "decisao"]

# Mesmo tipo da chave nos dois lados (evita ValueError no merge)
df_decisoes["no_solicitacao"] = pd.to_numeric(df_decisoes["no_solicitacao"], errors="coerce").astype("Int64")
df_gold["no_solicitacao"] = pd.to_numeric(df_gold["no_solicitacao"], errors="coerce").astype("Int64")

df_gold = df_gold.merge(df_decisoes[["no_solicitacao", "decisao"]], on="no_solicitacao", how="left")
print(f"Coluna 'decisao' adicionada. Gold: {len(df_gold)} linhas | {len(df_gold.columns)} colunas")

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 8, Finished, Available, Finished, False)

Coluna 'decisao' adicionada. Gold: 7451 linhas | 51 colunas


## 4. Verificações rápidas (preenchidos/nulos e colunas)

In [7]:
n_linhas = len(df_gold)
preenchidos = df_gold.count()
nulos = df_gold.isna().sum()
resumo = pd.DataFrame({
    "coluna": preenchidos.index,
    "preenchidos": preenchidos.values,
    "nulos": nulos.values,
    "% preenchido": (preenchidos / n_linhas * 100).round(1).values,
}).sort_values("% preenchido", ascending=False)
print(f"Total: {n_linhas} linhas | {len(df_gold.columns)} colunas")
display(resumo)

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 9, Finished, Available, Finished, False)

Total: 7451 linhas | 51 colunas


SynapseWidget(Synapse.DataFrame, cc30de56-41f3-4383-a0a3-7882ee5ede78)

In [8]:
# Colunas a dropar (não usadas nos painéis de BI)
COLUNAS_A_DROPAR = [
    "qual_e_o_uso_do_local_a_ser_demolido",
    "o_uso_do_local_e_misto",
    "qual_e_o_uso_do_local",
    "o_uso_do_local_sera_misto",
    "qual_sera_o_uso_do_local",
    "o_uso_da_edificacao_a_ser_conservada_e_misto",
    "o_uso_da_edificacao_a_ser_regularizada_e_misto",
    "uso_do_local",
    "area_total_a_ser_construida",
    "area_total_da_movimentacao_de_terra_m2",
    "area_m2",
    "area_total_a_ser_reconstruida_m2",
    "metragem_do_uso_a_ser_demolida_m2",
    "area_total_a_ser_conservada",
    "area_total_da_edificacao_m2",
    "area_total_a_ser_reformada_m2",
    "area_total_a_ser_regularizada",
    "anexo_do_comprovante_de_residencia",
    "setor_1_a_37",
    "cnpj",
    "cpf",
    "nome",
]

# Aplicar drop em df_gold (que já tem a coluna decisao)
_cols_dropar = [c for c in COLUNAS_A_DROPAR if c in df_gold.columns]
df_gold = df_gold.drop(columns=_cols_dropar)
print(f"Colunas dropadas: {len(_cols_dropar)} | Restantes: {len(df_gold.columns)}")

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 10, Finished, Available, Finished, False)

Colunas dropadas: 22 | Restantes: 29


In [9]:
df_gold

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 11, Finished, Available, Finished, False)

,no_solicitacao,servico,status_fluxo,data_finalizacao,data_criacao,solicitante,data_criacao_etapa,bairro,cep,cidade,...,inscricao_fiscal,razao_social,assunto,data_de_emissao,data_de_validade,nome_completo,n_documento,status,endereco,decisao
0,561871,2° VIA DE HABITE-SE,Cancelado,2024-05-07 10:49:25.567000+00:00,2024-05-07 10:39:46+00:00,FELIPE SILVA BALBINO,NaT,,,,...,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
1,564081,2° VIA DE HABITE-SE,Finalizado,2024-09-30 14:06:06.922000+00:00,2024-05-10 15:45:20+00:00,VIVIANE CELY MIGUEL,2024-07-30 18:47:31+00:00,Colônia,09405-520,Ribeirão Pires,...,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
2,565419,2° VIA DE HABITE-SE,Finalizado,2024-05-14 19:53:41.834000+00:00,2024-05-14 14:36:36+00:00,VIVIANE CELY MIGUEL,2024-05-14 17:19:29+00:00,Colônia,09405-520,Ribeirão Pires,...,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
3,568003,2° VIA DE HABITE-SE,Finalizado,2024-05-20 13:28:30.816000+00:00,2024-05-17 14:38:48+00:00,ARTUR TIZO,2024-05-17 14:53:11+00:00,Santo Antônio,09531-110,São Caetano do Sul,...,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
4,568923,2° VIA DE HABITE-SE,Finalizado,2024-05-20 19:09:02.594000+00:00,2024-05-20 13:16:59+00:00,ERIC PASSARELLI DESTEFANE,2024-05-20 14:17:40+00:00,Jardim Mauá,09340-190,Mauá,...,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7446,845114,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",Cancelado,2025-10-09 15:56:25.285000+00:00,NaT,EDUARDO MOBILE,NaT,NaN,NaN,NaN,...,,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
7447,846441,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",Em atendimento,2026-01-30 18:48:29.646000+00:00,2025-10-13 15:27:02+00:00,EDUARDO MOBILE,2025-10-23 18:27:09+00:00,NaN,NaN,NaN,...,19.006.109,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
7448,850791,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",Cancelado,2025-10-22 16:22:55.154000+00:00,NaT,WELLINGTON RAMOS DA SILVA,NaT,NaN,NaN,NaN,...,,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,
7449,851897,"TRANSFERÊNCIA, BAIXA E ASSUNÇÃO DE RESPONSABIL...",Cancelado,2025-10-24 10:58:33.625000+00:00,NaT,WELLINGTON RAMOS DA SILVA,NaT,NaN,NaN,NaN,...,,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,


## 5. Escrever tabela gold_maua_pl_urbano

In [10]:
df_gold_spark = spark.createDataFrame(df_gold)
df_gold_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(NOME_TABELA_GOLD)
print("Gold salvo (Fabric – Tables):", NOME_TABELA_GOLD)

StatementMeta(, 324d9f4b-3491-45ae-8109-56ff585ff863, 12, Finished, Available, Finished, False)

Gold salvo (Fabric – Tables): gold_maua_pl_urbano
